# A simple livability index for Buenos Aires

Dimensions:
- Accessibility and mobility
- Green spaces and recreation
- Public services and amenities
- Safety and security
- Economic opportunities

## Step 2: deriving district metrics

In [0]:
import os
import osmnx as ox
import pandas as pd
import geopandas as gpd

from sklearn.preprocessing import MinMaxScaler

CRS = "EPSG:9498"

In [0]:
def characterize_district(place, folderin, crs):
    admin_district = gpd.read_parquet(folderin + "/admin_district.parquet")
    public_transport = gpd.read_parquet(folderin + "/public_transport.parquet")
    parks_and_recreation = gpd.read_parquet(folderin + "/parks_and_recreation.parquet")
    healthcare = gpd.read_parquet(folderin + "/healthcare.parquet")
    education = gpd.read_parquet(folderin + "/education.parquet")
    emergency_services = gpd.read_parquet(folderin + "/emergency_services.parquet")
    commerce = gpd.read_parquet(folderin + "/commerce.parquet")
    employment = gpd.read_parquet(folderin + "/employment.parquet")

    district_area = (admin_district.to_crs(crs).geometry.area / 1000 ** 2)[0]

    data = {
        "place": place,
        "public_transport_density": len(public_transport) / district_area,
        "parks_and_recreation_density": len(parks_and_recreation) / district_area,
        "parks_and_recreation_coverage": parks_and_recreation.to_crs(crs).area.sum() / (district_area * 1000 ** 2),
        "healthcare_density": len(healthcare) / district_area,
        "education_density": len(education) / district_area,
        "emergency_services_density": len(emergency_services) / district_area,
        "commerce_density": len(commerce) / district_area,
        "employment_density": len(employment) / district_area,
    }

    return pd.DataFrame([data])

In [0]:
district_profiles = []

for i in range(1, 16):
    place = f"Comuna {i}, CABA"
    folderin = f"data/external/comuna{i:02}"

    district_profiles.append(characterize_district(place, folderin, CRS))

district_profiles = pd.concat(district_profiles)

In [0]:
folderout = "data/interim"

if not os.path.exists(folderout):
        os.makedirs(folderout)

district_profiles.to_csv(folderout + "/district_profiles.csv", index=False)